### Programming for Biomedical Informatics
#### Week 3 - Data Integration & Summary Analysis

This week we're first going to practice using a range of the eUtilities end-points to search, fetch, and link data.

In [1]:
# Preliminaries
from Bio import Entrez
import urllib.request
import json
import xml.etree.ElementTree as ET


# load my API key from the file
with open('../../bio_api_keys/ncbi.txt', 'r') as file:
    api_key = file.read().strip()

with open('../../bio_api_keys/ncbi_email.txt', 'r') as file:
    email = file.read().strip()

Entrez.api_key = api_key
Entrez.email = email

In [2]:
# Step One - Example Using eInfo

# let's use biopython Entrez module to do an eInfo query
# this will tell us what databases are available
handle = Entrez.einfo()

# read the handle
record = Entrez.read(handle)

# print the record in an easy to read format
# print(json.dumps(record, indent=4))

# print out some useful information about each of these databases
# NB this is incredibly useful but long so best to do it for a particular database
# for db in record['DbList']:
#     print(f"Database: {db}")
#     # get the database info
#     db_info = Entrez.read(Entrez.einfo(db=db))
#     # print the database info
#     print(json.dumps(db_info, indent=4))

# lets do this just for the gene database
gene_info = Entrez.read(Entrez.einfo(db='gene'))

# this is a nice way to print out nested XML structures in a readable way
print(json.dumps(gene_info, indent=4))

{
    "DbInfo": {
        "DbName": "gene",
        "MenuName": "Gene",
        "Description": "Gene database",
        "DbBuild": "Build250930-2320m.1",
        "Count": "89762485",
        "LastUpdate": "2025/10/01 12:42",
        "FieldList": [
            {
                "Name": "ALL",
                "FullName": "All Fields",
                "Description": "All terms from all searchable fields",
                "TermCount": "1614968260",
                "IsDate": "N",
                "IsNumerical": "N",
                "SingleToken": "N",
                "Hierarchy": "N",
                "IsHidden": "N"
            },
            {
                "Name": "UID",
                "FullName": "UID",
                "Description": "Unique number assigned to a gene record",
                "TermCount": "0",
                "IsDate": "N",
                "IsNumerical": "Y",
                "SingleToken": "Y",
                "Hierarchy": "N",
                "IsHidden": "Y"
          

In [13]:
# Step Two - Example using eSearch to search the gene database for a particular gene
# search for the gene 'BRCA1'
handle = Entrez.esearch(db='gene', term='BRCA1', retmode='xml')

#retreive the record
record = Entrez.read(handle)

# find the count of records
count = record['Count']
print(f"Number of records found: {count}")

# why are there so many?
# Did not specify human or species
# What is gene? We did not specify to get just genes, it will get any mention of 'BRCAI' "All fields"

Number of records found: 34471


In [14]:
record

{'Count': '34471', 'RetMax': '20', 'RetStart': '0', 'IdList': ['143861250', '143856652', '143839474', '143832220', '143832066', '143824892', '143822976', '143819396', '143809704', '143804890', '143787858', '143786642', '143784246', '143687742', '143683624', '143678056', '143677020', '143675770', '143668294', '143666888'], 'TranslationSet': [], 'TranslationStack': [{'Term': 'BRCA1[All Fields]', 'Field': 'All Fields', 'Count': '34471', 'Explode': 'N'}, 'GROUP'], 'QueryTranslation': 'BRCA1[All Fields]'}

In [15]:
# print the record in an easy to read format
print(json.dumps(record, indent=4))

# NB the [All Fields] search is a broad search that will return many results

{
    "Count": "34471",
    "RetMax": "20",
    "RetStart": "0",
    "IdList": [
        "143861250",
        "143856652",
        "143839474",
        "143832220",
        "143832066",
        "143824892",
        "143822976",
        "143819396",
        "143809704",
        "143804890",
        "143787858",
        "143786642",
        "143784246",
        "143687742",
        "143683624",
        "143678056",
        "143677020",
        "143675770",
        "143668294",
        "143666888"
    ],
    "TranslationSet": [],
    "TranslationStack": [
        {
            "Term": "BRCA1[All Fields]",
            "Field": "All Fields",
            "Count": "34471",
            "Explode": "N"
        },
        "GROUP"
    ],
    "QueryTranslation": "BRCA1[All Fields]"
}


In [ ]:
# Step Three - Example using eSummary to get some metadata about the first 5 of these records
# this time we're going to do this using urllib.requests to show how tou can do this independenly of biopython
# we will request xml and parse that using ElementTree

# get the first 5 ids
ids = record['IdList'][:5]

eUtils_base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
eSummary = "esummary.fcgi"

# don't forget to add the API key and email
#for each id in the list pull the summary
for id in ids:
    url = f"{eUtils_base}{eSummary}?db=gene&id={id}&api_key={api_key}&email={email}"
    with urllib.request.urlopen(url) as response:
        xml = response.read()
        root = ET.fromstring(xml)
        # find the <Organsim> tag
        organism = root.find('DocumentSummarySet/DocumentSummary/Organism/').text
        name = root.find('DocumentSummarySet/DocumentSummary/').text
        print(id,name,organism)

# OK what's going on here?
# Here we see we've got gene names not expected, this means that somewhere in the ids shown, the BRCA1 term is mentioned, no specifically returning BRCA1[gene]

143861250 LOC143861250 Tasmannia lanceolata
143856652 LOC143856652 Tasmannia lanceolata
143839474 BABAM2 Paroedura picta
143832220 UIMC1 Paroedura picta
143832066 BAP1 Paroedura picta


In [17]:
# Step Four - Lets look again but add the [Gene] field to the search
handle = Entrez.esearch(db='gene', term='BRCA1[Gene]', retmode='xml')

#retreive the record
record = Entrez.read(handle)

# find the count of records
count = record['Count']
print(f"Number of records found: {count}")

# print the record in an easy to read format
print(json.dumps(record, indent=4))

Number of records found: 566
{
    "Count": "566",
    "RetMax": "20",
    "RetStart": "0",
    "IdList": [
        "143892821",
        "143825099",
        "143765407",
        "143687345",
        "143409333",
        "672",
        "12189",
        "497672",
        "403437",
        "373983",
        "827854",
        "353120",
        "399391",
        "712634",
        "449497",
        "554178",
        "100049662",
        "101081937",
        "129469801",
        "129017314"
    ],
    "TranslationSet": [],
    "TranslationStack": [
        {
            "Term": "BRCA1[Gene]",
            "Field": "Gene",
            "Count": "566",
            "Explode": "N"
        },
        "GROUP"
    ],
    "QueryTranslation": "BRCA1[Gene]"
}


In [ ]:
# get the first 5 ids
ids = record['IdList'][:5]

eUtils_base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
eSummary = "esummary.fcgi"

# don't forget to add the API key and email
#for each id in the list pull the summary
for id in ids:
    url = f"{eUtils_base}{eSummary}?db=gene&id={id}&api_key={api_key}&email={email}"
    with urllib.request.urlopen(url) as response:
        xml = response.read()
        root = ET.fromstring(xml)
        # find the <Organsim> tag
        organism = root.find('DocumentSummarySet/DocumentSummary/Organism/').text
        name = root.find('DocumentSummarySet/DocumentSummary/').text
        print(id,name,organism)

# Ah that's better, now we've got BRCA1 but from lots of different organisms
# We can see we're getting non homo-sapien orgamisms as well as the desired human gene. The all fields will match all occurences in this way, not just genes, but refernces to the term

143892821 BRCA1 Tasmannia lanceolata
143825099 BRCA1 Paroedura picta
143765407 BRCA1 Ranitomeya variabilis
143687345 BRCA1 Tamandua tetradactyla
143409333 Brca1 Callospermophilus lateralis


In [ ]:
# Step Five - Finally, let's get this right

# search for the gene 'BRCA1' in the human genome using eSearch
handle = Entrez.esearch(db='gene', term='BRCA1[Gene], human[Organism]', retmode='xml') # notes the xml return type

#retreive the record
record = Entrez.read(handle) # accepts the xml handle returned from the eSearch and python dictionary that is easier to handle

# find the count of records
count = record['Count']
print(f"Number of records found: {count}")

# print the record in an easy to read format
print(json.dumps(record, indent=4))

# get the only id
id = record['IdList'][0]

# The translation set below shows from a set dictionary that human[Organism] is recognised and translated to the proper Homo sapiens['Organism']

Number of records found: 1
{
    "Count": "1",
    "RetMax": "1",
    "RetStart": "0",
    "IdList": [
        "672"
    ],
    "TranslationSet": [
        {
            "From": ", human[Organism]",
            "To": "\"Homo sapiens\"[Organism]"
        }
    ],
    "TranslationStack": [
        {
            "Term": "BRCA1[Gene]",
            "Field": "Gene",
            "Count": "566",
            "Explode": "N"
        },
        {
            "Term": "\"Homo sapiens\"[Organism]",
            "Field": "Organism",
            "Count": "360293",
            "Explode": "Y"
        },
        "AND"
    ],
    "QueryTranslation": "BRCA1[Gene] AND \"Homo sapiens\"[Organism]"
}


In [63]:
# now use eSummary to get the metadata for this gene
url = f"{eUtils_base}{eSummary}?db=gene&id={id}&api_key={api_key}&email={email}" #Here we specify our url and endpoints in the same way
response = urllib.request.urlopen(url)
root = ET.fromstring(response.read()) # This received the raw xml bytes from teh url request and formats into xml document to be handled using E.T.
organism = root.find('DocumentSummarySet/DocumentSummary/Organism/').text
name = root.find('DocumentSummarySet/DocumentSummary/').text
print(id,name,organism)

# Notes:
# Playing with json format is important to decipher
# The Rootfind gives a series of Document summary fields, so it's a way of getting through taxonomy
# Parsing through Json and XML is important

672 BRCA1 Homo sapiens


In [68]:
print(root.text)

In [71]:
print(root.tag, root.attrib, len(root), repr(root.text))
print('root.text: ', root.text)
print(ET.tostring(root, encoding="unicode"))
print('------------------------------------------')
dss = root.find('DocumentSummarySet')
print(dss.tag, dss.attrib, len(dss), repr(dss.text))
print('dss_1.text: ', dss.text)
print(ET.tostring(dss, encoding='unicode'))
print('------------------------------------------')
dss_2 = root.find('DocumentSummarySet/DocumentSummary')
print(dss_2.tag, dss_2.attrib, len(dss_2), repr(dss_2.text))
print('dss_2.text: ',dss_2.text)
print(ET.tostring(dss_2, encoding='unicode'))


eSummaryResult {} 1 '\n'
root.text:  

<eSummaryResult>
<DocumentSummarySet status="OK">
<DbBuild>Build250930-2320m.1</DbBuild>

<DocumentSummary uid="672">
	<Name>BRCA1</Name>
	<Description>BRCA1 DNA repair associated</Description>
	<Status>0</Status>
	<CurrentID>0</CurrentID>
	<Chromosome>17</Chromosome>
	<GeneticSource>genomic</GeneticSource>
	<MapLocation>17q21.31</MapLocation>
	<OtherAliases>BRCAI, BRCC1, BROVCA1, FANCS, IRIS, PNCA4, PPP1R53, PSCP, RNF53</OtherAliases>
	<OtherDesignations>breast cancer type 1 susceptibility protein|BRCA1/BRCA2-containing complex, subunit 1|Fanconi anemia, complementation group S|RING finger protein 53|breast and ovarian cancer susceptibility protein 1|breast cancer 1, early onset|early onset breast cancer 1|protein phosphatase 1, regulatory subunit 53</OtherDesignations>
	<NomenclatureSymbol>BRCA1</NomenclatureSymbol>
	<NomenclatureName>BRCA1 DNA repair associated</NomenclatureName>
	<NomenclatureStatus>Official</NomenclatureStatus>
	<Mim>
		<int>

In [42]:
ET.dump(root)

<eSummaryResult>
<DocumentSummarySet status="OK">
<DbBuild>Build250930-2320m.1</DbBuild>

<DocumentSummary uid="672">
	<Name>BRCA1</Name>
	<Description>BRCA1 DNA repair associated</Description>
	<Status>0</Status>
	<CurrentID>0</CurrentID>
	<Chromosome>17</Chromosome>
	<GeneticSource>genomic</GeneticSource>
	<MapLocation>17q21.31</MapLocation>
	<OtherAliases>BRCAI, BRCC1, BROVCA1, FANCS, IRIS, PNCA4, PPP1R53, PSCP, RNF53</OtherAliases>
	<OtherDesignations>breast cancer type 1 susceptibility protein|BRCA1/BRCA2-containing complex, subunit 1|Fanconi anemia, complementation group S|RING finger protein 53|breast and ovarian cancer susceptibility protein 1|breast cancer 1, early onset|early onset breast cancer 1|protein phosphatase 1, regulatory subunit 53</OtherDesignations>
	<NomenclatureSymbol>BRCA1</NomenclatureSymbol>
	<NomenclatureName>BRCA1 DNA repair associated</NomenclatureName>
	<NomenclatureStatus>Official</NomenclatureStatus>
	<Mim>
		<int>113705</int>
	</Mim>
	<GenomicInfo>
		<

In [ ]:
# Step Six - Use eFetch to get the full record

# lets modify the code above to use eFetch to get the full record
eFetch = "efetch.fcgi"

url = f"{eUtils_base}{eFetch}?db=gene&id={id}&api_key={api_key}&email={email}"
response = urllib.request.urlopen(url)
xml = response.read()

# print the xml
print(xml.decode('utf-8'))

# Notes:
# generif are functions to show interactions with other genes

In [ ]:
# Step Seven - Use eLink to get the associated nucleotide sequence

# lets go back to using the Entrez module to get the sequence
# get the link specifically for the refseq gene nucleotide sequence
links = Entrez.read(Entrez.elink(dbfrom='gene', id=id, linkname='gene_nuccore_refseqgene'))

# get the id of the nucleotide sequence
nuccore_id = links[0]['LinkSetDb'][0]['Link'][0]['Id']

# get the sequence
handle = Entrez.efetch(db='nuccore', id=nuccore_id, rettype='fasta', retmode='text')
sequence = handle.read()

# print the sequence
print(sequence)

# In theory the way to pull out relationships between different objects of in the NCBI
# RefSeq is gold standard for reference genes

